# Q3 누락 진단: 기존 후보 / 사람 정답 칸 / 사람 정답 박스 확대

파일: `lvlm_q3_gt_crop_diagnostic_dev_colab.ipynb`

**L4/A100 → 기본 설정 유지 → 모두 실행 → Drive 연결 허용.**
실제 추론용 코랩입니다. 준비/CPU 검증과 실제 효과는 별개이며, 실제 효과는 실행 후 확인합니다.

- 완료한25장 실제 표시를 내장했습니다(박스24/판단불가1). 다시 표시하거나 결과 ZIP을 입력할 필요 없습니다.
- 이전 후보 확대를 실행했고 위치 정답이 있는 동일20장만 세 조건으로 비교합니다. 판단불가1/후보보류1/이전경로밖3은 제외하고 이유를 기록합니다.
- 기존 후보 칸 확대 / 사람 정답 칸 확대 / 정답 박스에 각 방향25% 여백을 붙인 확대. 정상 참조+원본+확대판3이미지, 동일3/3/2타일·80토큰·조건당1회입니다.
- **최대60회 LVLM, DINO/재학습/Q1/Q2/gold0.** 기존 판별기·가림 실험을 다시 돌리지 않습니다.
- **정답을 입력하는 진단 실험입니다. 최종 Agent나 독립 holdout 점수가 아닙니다.** 정상 사진이 없어 정상 오탐률은 평가하지 않습니다.
- 정답 칸 조건으로 위치 선정 문제를, 정답 박스 조건으로 크기/문맥 문제를 살핍니다. 원본9칸 좌표로 답하며 무효는 오답으로 유지합니다.
- 확대판과 질문에는 정답 칸 번호·박스 좌표를 적지 않습니다. 세 조건은 완전히 같은 질문을 사용하며, 정답은 확대 부위 선택에만 씁니다.
- 같은 코드/입력/모델/폴더면 성공 기록에서 재개합니다. 실행 오류만 재시도합니다. 결과가 틀렸다고 반복하지 않습니다.

출력: `q3_gt_crop_diagnostic_dev20_results.zip`. 기존 원본 데이터 ZIP 두 개를 자동 검색합니다.

In [ ]:
RUN_FOLDER = "lvlm_insplad_results/q3_gt_crop_diagnostic_dev20_v1"
SUP_ZIP_PATH = ""
UNSUP_ZIP_PATH = ""

In [ ]:
import base64,hashlib,io,json,sys,zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
DRIVE=Path('/content/drive/MyDrive')
SOURCE_SHA256='cf89e774f465a46055a1f33103a48f0aac8e65e3e5c05f1cbe0d819edea830a2'
SOURCE_MANIFEST={'scripts/build_explainable_qa_gold.py': '9acdd42e87c15f9da2ba9bb7c3f89d9e77c42d774a8a8d38f6e783ea544d4f0a', 'scripts/build_gold_v2_dev_set.py': '31c1f17112eea9b279466cdeaa6fdafb43c450be2d6b268136f1b8b8a429f66a', 'scripts/build_q3_gt_crop_diagnostic_notebook.py': '72365580f34be2680752545303aae83ebe9a3ab5aa9ed3d6f74039bd4ee27ad1', 'scripts/build_retrieved_refs.py': '8f27f2aec2d7eecba622f651c2255a4da98ec435bbf9fd6fe2eedca96ddbed93', 'scripts/dev_missing_location_annotation.py': '3632cb9a32e304acfb11d1e610c47fb0536b453a77dc52ba82383b9baa24b680', 'scripts/dev_part_crop_annotation.py': '39de07b82f31778bfb0a94eac89265dd200296f7fa97e2c60ff6308b180682ea', 'scripts/loc_box_annotation.py': '1d839d6a6b10dda3c4c1d3c7ea16aa17e5d6b072f2df90d7e36c4d64fe316cff', 'scripts/q3_multiregion.py': '9eac922fd3ea6cb9164aa8ff8e511ad9a10e665da871e548192c3a53363a2063', 'scripts/run_dev_prompt_diagnostic.py': '76400860bc603002e6c3aa295b212b02bb2f78fd47b0d52f56c34ec2bd8dc8ab', 'scripts/run_gold_baselines.py': '978cb258c7412a767de2b5f7ee9604e726c29dc41922d656084d751d32869575', 'scripts/run_gold_v2_baselines.py': '72585045062bfc5f5ec5243ca50b34ba7fe4bb871f400bd734712354caec9a2a', 'scripts/run_gold_v2_consistency_metrics.py': '283ab1b1c2c1355895a0a047b9da5cfaef9ed67bf55f0108a0cfd6912e4f2dc2', 'scripts/run_gold_v2_dev_baselines.py': '809eca7a79e59f546107409f2a22f2bd1fda2cef3c77b8ea091b0cd401770fa9', 'scripts/run_gold_v2_q23_candidate.py': '19357ad3507dffd33ff5cd97d7febe36c87ab80a255e09398e380770c8d68499', 'scripts/run_gold_v2_q23_zoom_prompt_dev.py': 'cc8b7937fa75250b99af39ea5fa115cae06cb6f5aa470d0c28f417c4ed246d8a', 'scripts/run_gold_v2_q2_q3_agent_dev.py': '0dec0fd461b115de229913463ed7495552bcfb37a548dea0662f212800d695c8', 'scripts/run_gold_v2_q2_q3_refined_dev.py': '577837b328e7b1a9c8b0ea5a06638a798a9f043d84d58f7e085302b531ba8682', 'scripts/run_gold_v2_reference_ablation.py': 'fff6825808e7b9869c42ec2f54438cb98917c442fcdd2e6797d4554daaed9234', 'scripts/run_gold_v2_verify_agent.py': '0963c6c0c0e19c3bffd7fd7b68dcbd54ade6c51ba64bfd04f41fb0fafe5f966a', 'scripts/run_missing_component_probe.py': 'fcb36a2c74d5afaf2f9a1ffdd08a05660b37144cde8ed87d89ba55174b1adfc3', 'scripts/run_q2_conflict_evidence_dev.py': '52ccb22309b3771ef370bdfc271256c8eb39c603f6833900b72e6dd1a9f381c2', 'scripts/run_q3_gt_crop_diagnostic_dev.py': 'a134d9a1e9596ba10bccb9eb4eb29d065b7357f72e41ee5eea694c9eecabf741', 'scripts/run_q3_missing_occlusion_dev.py': '30d2faabdcecd315724f208e4131fb4dd3bb4c847bc5b05ce034268070298227', 'scripts/run_selective_missing_expert_agent_dev.py': 'd7dde258c09b6c8d4408226d2d1927616e240185033aabd985d14ab38f6f3288', 'src/insplad_benchmark/__init__.py': '5d9678c34324691fa8d7d5b507501945bb6eafd32e53ef6dedf1979e005cca49', 'src/insplad_benchmark/dataset.py': 'f724c3c5a76f4caa8fa86ac3d3fddef8fed0c466943f979b815f380fbd46652c', 'src/insplad_benchmark/hf_adapters.py': 'dbf178f1a6263c371fc8c7db192a33bdd252910b63267583d5bc24db11f684ab', 'src/insplad_benchmark/metrics.py': 'bb9386501a38031f3d5993744f02a4a1beeb3610bf2a1daa659100511f04cc9d', 'inputs/dataset.json': 'e4f36d45032b622e15c46d4ee38483a444bf1cf783b7efc61cdf055716255ec5', 'inputs/annotations.json': '9a9e71b995174288e3507e0d60f97fd155abb2149de2bf4ee0feb10fe02d52d3', 'inputs/annotation_manifest.json': '10137a0017c108f51984d71da3077c81af4c9f54bdfcfab392a62c1d51c77644'}
SOURCE_BYTES=base64.b64decode('...')
assert hashlib.sha256(SOURCE_BYTES).hexdigest()==SOURCE_SHA256
CODE=Path('/content/q3_gt_crop_'+SOURCE_SHA256[:12])
with zipfile.ZipFile(io.BytesIO(SOURCE_BYTES)) as archive:
    assert set(archive.namelist())==set(SOURCE_MANIFEST)
    for name in archive.namelist():
        dest=CODE/name;content=archive.read(name)
        assert dest.resolve().is_relative_to(CODE.resolve())
        assert hashlib.sha256(content).hexdigest()==SOURCE_MANIFEST[name]
        if dest.exists() and dest.read_bytes()!=content:raise RuntimeError('내장 코드와 다른 파일; 새 런타임을 사용하세요.')
        dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(content)
sys.path[:0]=[str(CODE/'scripts'),str(CODE/'src')]
import run_q3_gt_crop_diagnostic_dev as experiment
DATASET=CODE/'inputs/dataset.json';OUT=DRIVE/RUN_FOLDER
assert OUT.name.startswith(experiment.PREFIX)
print('진단 전용: 동일20장 ×3조건=최대60회. SOURCE',SOURCE_SHA256)

In [ ]:
import subprocess
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory<20*1024**3:
    raise RuntimeError('L4/A100 VRAM20GB 이상 런타임을 선택하세요.')
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.14.0','accelerate==1.14.0',
    'sentencepiece==0.2.2','numpy>=1.26','pillow>=10','timm'],check=True)
def find_zip(name,size,override):
    if override:
        path=Path(override)
        if not path.is_file() or path.stat().st_size!=size:raise FileNotFoundError(str(path))
        return path
    for path in [DRIVE/name,DRIVE/'lvlm_insplad_data'/name]:
        if path.is_file() and path.stat().st_size==size:return path
    matches=[p for p in DRIVE.rglob(name) if p.is_file() and p.stat().st_size==size]
    if not matches:raise FileNotFoundError(name)
    return sorted(matches)[0]
SUP=find_zip('supervised_fault_classification.zip',874753167,SUP_ZIP_PATH)
UNSUP=find_zip('unsupervised_anomaly_detection.zip',1170967719,UNSUP_ZIP_PATH)
from huggingface_hub import snapshot_download
snapshot=snapshot_download(experiment.old.prior.MODEL_ID,revision=experiment.old.prior.MODEL_REVISION)
command=[sys.executable,str(CODE/'scripts/run_q3_gt_crop_diagnostic_dev.py'),
    '--dataset',str(DATASET),'--sup-zip',str(SUP),'--unsup-zip',str(UNSUP),
    '--model-path',snapshot,'--out-dir',str(OUT)]
LOG=DRIVE/'q3_gt_crop_diagnostic_dev20_console.log'
process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with LOG.open('a',encoding='utf-8') as log:
    for line in process.stdout:print(line,end='');log.write(line);log.flush()
RETURN_CODE=process.wait();print('RUNNER_RETURN_CODE',RETURN_CODE)

In [ ]:
from google.colab import files
RESULT=DRIVE/'q3_gt_crop_diagnostic_dev20_results.zip'
with zipfile.ZipFile(RESULT,'w',zipfile.ZIP_DEFLATED) as archive:
    if OUT.exists():
        for path in sorted(OUT.rglob('*')):
            if path.is_file() and '_images' not in path.relative_to(OUT).parts:
                archive.write(path,'run/'+path.relative_to(OUT).as_posix())
    archive.writestr('source_bundle.zip',SOURCE_BYTES)
    archive.writestr('source_manifest.json',json.dumps(SOURCE_MANIFEST,indent=2))
    archive.writestr('runner_return_code.json',json.dumps({'return_code':RETURN_CODE}))
    if LOG.exists():archive.write(LOG,'console.log')
status=json.loads((OUT/'run_status.json').read_text()) if (OUT/'run_status.json').exists() else {'complete':False}
print('RUN STATUS',status)
if RETURN_CODE or not status.get('complete'):print('실행 오류가 있습니다. ZIP의 원시 기록을 확인해야 합니다.')
elif status.get('invalid_new_outputs'):print('평가 완료이나 무효 출력은 오답으로 포함했습니다.')
print('진단 전용 / Agent 점수 아님. ZIP',RESULT,'SHA256',hashlib.sha256(RESULT.read_bytes()).hexdigest())
files.download(str(RESULT))